# Train Fingerspelling Classifier

This notebook trains the 26-letter classifier. There are 4 different architectures (RF, MLP, XGBoost, and SVM).


In [3]:
import os
import json
import pickle
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from xgboost import XGBClassifier


BASE_DIR = os.path.abspath('')
DATA_PATH = os.path.join(BASE_DIR, "data.pickle")
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_PATH = os.path.join(BASE_DIR, "results.json")
os.makedirs(MODELS_DIR, exist_ok=True)

SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
EPOCHS = 200
PATIENCE = 20
BATCH_SIZE = 64
JITTER_STD = 0.01
LABEL_SMOOTHING = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def load_data():
    with open(DATA_PATH, "rb") as f:
        data_dict = pickle.load(f)

    data = np.asarray(data_dict["data"], dtype=np.float32)
    labels = np.asarray(data_dict["labels"])

    if data.ndim != 2 or data.shape[1] != 63:
        raise ValueError(f"Expected data shape (n_samples, 63), got {data.shape}")

    le = LabelEncoder()
    y = le.fit_transform(labels)

    return data, y, le


class LandmarkDataset(Dataset):
    def __init__(self, x, y, augment=False, jitter_std=0.01):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment
        self.jitter_std = jitter_std

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        sample = self.x[idx].clone()
        if self.augment:
            sample = sample + torch.randn_like(sample) * self.jitter_std
        return sample, self.y[idx]


class MLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, num_classes, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for width in hidden_layers:
            layers.append(nn.Linear(prev, width))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = width
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


@dataclass
class MLPConfig:
    hidden_layers: tuple
    dropout: float
    lr: float
    weight_decay: float


RF_CONFIGS = [
    {"n_estimators": 200, "max_depth": None, "min_samples_split": 2},
    {"n_estimators": 400, "max_depth": 20, "min_samples_split": 2},
    {"n_estimators": 600, "max_depth": 25, "min_samples_split": 2},
    {"n_estimators": 800, "max_depth": 30, "min_samples_split": 4},
]

SVM_CONFIGS = [
    {"kernel": "rbf", "C": 1.0, "gamma": "scale"},
    {"kernel": "rbf", "C": 10.0, "gamma": "scale"},
    {"kernel": "rbf", "C": 25.0, "gamma": "scale"},
    {"kernel": "poly", "C": 10.0, "degree": 3, "gamma": "scale"},
]

XGB_CONFIGS = [
    {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 500, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 700, "subsample": 0.85, "colsample_bytree": 0.85},
    {"max_depth": 8, "learning_rate": 0.03, "n_estimators": 900, "subsample": 0.9, "colsample_bytree": 0.9},
    {"max_depth": 10, "learning_rate": 0.03, "n_estimators": 1200, "subsample": 0.9, "colsample_bytree": 0.9},
]

MLP_CONFIGS = [
    MLPConfig(hidden_layers=(128, 128), dropout=0.15, lr=1e-3, weight_decay=1e-4),
    MLPConfig(hidden_layers=(256, 256), dropout=0.20, lr=8e-4, weight_decay=1e-4),
    MLPConfig(hidden_layers=(512, 256), dropout=0.25, lr=8e-4, weight_decay=5e-5),
    MLPConfig(hidden_layers=(512, 512, 256), dropout=0.30, lr=5e-4, weight_decay=5e-5),
]


def train_and_eval_sklearn(name, model, x_train, y_train, x_test, y_test):
    print(f"\n[{name}] training...")
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    acc = accuracy_score(y_test, preds)
    print(f"[{name}] test accuracy: {acc * 100:.2f}%")
    out_path = os.path.join(MODELS_DIR, f"{name}.pkl")
    with open(out_path, "wb") as f:
        pickle.dump({"model": model}, f)
    print(f"[{name}] saved: {out_path}")
    return acc


def train_and_eval_xgb(name, params, x_train, y_train, x_val, y_val, x_test, y_test, num_classes):
    print(f"\n[{name}] training...")
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=PATIENCE,
        **params,
    )
    model.fit(
        x_train,
        y_train,
        eval_set=[(x_val, y_val)],
        verbose=False,
    )
    preds = model.predict(x_test)
    acc = accuracy_score(y_test, preds)
    print(f"[{name}] best_iteration: {getattr(model, 'best_iteration', None)}")
    print(f"[{name}] test accuracy: {acc * 100:.2f}%")
    out_path = os.path.join(MODELS_DIR, f"{name}.json")
    model.save_model(out_path)
    print(f"[{name}] saved: {out_path}")
    return acc


def train_and_eval_mlp(name, config: MLPConfig, x_train, y_train, x_val, y_val, x_test, y_test, num_classes):
    print(f"\n[{name}] training on {DEVICE}...")
    train_ds = LandmarkDataset(x_train, y_train, augment=True, jitter_std=JITTER_STD)
    val_ds = LandmarkDataset(x_val, y_val, augment=False)
    test_ds = LandmarkDataset(x_test, y_test, augment=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = MLP(63, config.hidden_layers, num_classes, dropout=config.dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

    best_val_loss = float("inf")
    best_state = None
    patience_left = PATIENCE

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)
            train_correct += (logits.argmax(dim=1) == yb).sum().item()
            train_total += xb.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                logits = model(xb)
                loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                val_correct += (logits.argmax(dim=1) == yb).sum().item()
                val_total += xb.size(0)

        train_loss /= max(train_total, 1)
        val_loss /= max(val_total, 1)
        train_acc = train_correct / max(train_total, 1)
        val_acc = val_correct / max(val_total, 1)
        scheduler.step(val_loss)

        print(
            f"[{name}] epoch {epoch:03d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} | lr {optimizer.param_groups[0]['lr']:.6f}"
        )

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {
                "model_state_dict": model.state_dict(),
                "config": config.__dict__,
            }
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{name}] early stopping triggered.")
                break

    if best_state is None:
        best_state = {
            "model_state_dict": model.state_dict(),
            "config": config.__dict__,
        }

    model.load_state_dict(best_state["model_state_dict"])
    model.eval()

    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            test_correct += (logits.argmax(dim=1) == yb).sum().item()
            test_total += xb.size(0)

    acc = test_correct / max(test_total, 1)
    print(f"[{name}] test accuracy: {acc * 100:.2f}%")

    out_path = os.path.join(MODELS_DIR, f"{name}.pt")
    torch.save(best_state, out_path)
    print(f"[{name}] saved: {out_path}")
    return acc


def main():
    print(f"Loading data from: {DATA_PATH}")
    print(f"Saving models to: {MODELS_DIR}")
    data, y, label_encoder = load_data()
    num_classes = len(label_encoder.classes_)
    print(f"Loaded {len(data)} samples with {data.shape[1]} features and {num_classes} classes")
    print(f"Classes: {list(label_encoder.classes_)}")

    x_train, x_test, y_train, y_test = train_test_split(
        data,
        y,
        test_size=TEST_SIZE,
        shuffle=True,
        stratify=y,
        random_state=SEED,
    )

    x_train_main, x_val, y_train_main, y_val = train_test_split(
        x_train,
        y_train,
        test_size=VAL_SIZE,
        shuffle=True,
        stratify=y_train,
        random_state=SEED,
    )

    print(f"Train: {len(x_train_main)}, Val: {len(x_val)}, Test: {len(x_test)}")

    results = []

    for i, cfg in enumerate(RF_CONFIGS, start=1):
        name = f"RF_{i}"
        model = RandomForestClassifier(
            n_estimators=cfg["n_estimators"],
            max_depth=cfg["max_depth"],
            min_samples_split=cfg["min_samples_split"],
            random_state=SEED,
            n_jobs=-1,
        )
        acc = train_and_eval_sklearn(name, model, x_train_main, y_train_main, x_test, y_test)
        results.append({"model": name, "accuracy": acc})

    for i, cfg in enumerate(SVM_CONFIGS, start=1):
        name = f"SVM_{i}"
        model = SVC(
            kernel=cfg["kernel"],
            C=cfg["C"],
            gamma=cfg.get("gamma", "scale"),
            degree=cfg.get("degree", 3),
            probability=False,
            random_state=SEED,
        )
        acc = train_and_eval_sklearn(name, model, x_train_main, y_train_main, x_test, y_test)
        results.append({"model": name, "accuracy": acc})

    for i, cfg in enumerate(XGB_CONFIGS, start=1):
        name = f"XGB_{i}"
        acc = train_and_eval_xgb(
            name,
            cfg,
            x_train_main,
            y_train_main,
            x_val,
            y_val,
            x_test,
            y_test,
            num_classes,
        )
        results.append({"model": name, "accuracy": acc})

    for i, cfg in enumerate(MLP_CONFIGS, start=1):
        name = f"MLP_{i}"
        acc = train_and_eval_mlp(
            name,
            cfg,
            x_train_main,
            y_train_main,
            x_val,
            y_val,
            x_test,
            y_test,
            num_classes,
        )
        results.append({"model": name, "accuracy": acc})

    results = sorted(results, key=lambda x: x["accuracy"], reverse=True)
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print("\nFinal ranking:")
    for row in results:
        print(f"{row['model']}: {row['accuracy'] * 100:.2f}%")
    print(f"Saved results to: {RESULTS_PATH}")


if __name__ == "__main__":
    main()

Loading data from: d:\Computerpreter\Fingerspelling\data.pickle
Saving models to: d:\Computerpreter\Fingerspelling\models
Loaded 25858 samples with 63 features and 26 classes
Classes: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '3', '4', '5', '6', '7', '8', '9']
Train: 16548, Val: 4138, Test: 5172

[RF_1] training...
[RF_1] test accuracy: 98.03%
[RF_1] saved: d:\Computerpreter\Fingerspelling\models\RF_1.pkl

[RF_2] training...
[RF_2] test accuracy: 98.16%
[RF_2] saved: d:\Computerpreter\Fingerspelling\models\RF_2.pkl

[RF_3] training...
[RF_3] test accuracy: 98.16%
[RF_3] saved: d:\Computerpreter\Fingerspelling\models\RF_3.pkl

[RF_4] training...
[RF_4] test accuracy: 98.09%
[RF_4] saved: d:\Computerpreter\Fingerspelling\models\RF_4.pkl

[SVM_1] training...
[SVM_1] test accuracy: 96.66%
[SVM_1] saved: d:\Computerpreter\Fingerspelling\models\SVM_1.pkl

[SVM_2] training...
[SVM_2] test accuracy: 98.20%
[SVM_2] saved: d:\

In [4]:
from collections import OrderedDict

def load_results():
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def load_all_models():
    models = OrderedDict()
    for fname in sorted(os.listdir(MODELS_DIR)):
        path = os.path.join(MODELS_DIR, fname)
        if fname.endswith(".pkl"):
            with open(path, "rb") as f:
                models[fname] = pickle.load(f)["model"]
        elif fname.endswith(".json"):
            model = XGBClassifier()
            model.load_model(path)
            models[fname] = model
        elif fname.endswith(".pt"):
            ckpt = torch.load(path, map_location="cpu")
            cfg = ckpt["config"]
            model = MLP(63, tuple(cfg["hidden_layers"]), 26, dropout=cfg["dropout"])
            model.load_state_dict(ckpt["model_state_dict"])
            model.to(DEVICE)
            model.eval()
            models[fname] = (model, ckpt)
    return models


def print_model_specs(name, model_obj):
    print(f"\n=== {name} ===")
    if isinstance(model_obj, tuple):
        model, ckpt = model_obj
        print("Type: MLP")
        print("Hidden layers:", ckpt["config"]["hidden_layers"])
        print("Dropout:", ckpt["config"]["dropout"])
        print("Learning rate:", ckpt["config"]["lr"])
        print("Weight decay:", ckpt["config"]["weight_decay"])
        print("Device:", DEVICE)
        print("Parameters:", sum(p.numel() for p in model.parameters()))
        return

    cls_name = model_obj.__class__.__name__
    print("Type:", cls_name)
    if cls_name == "RandomForestClassifier":
        print("n_estimators:", model_obj.n_estimators)
        print("max_depth:", model_obj.max_depth)
        print("min_samples_split:", model_obj.min_samples_split)
        print("n_features_in_:", getattr(model_obj, "n_features_in_", "?"))
        print("classes_:", list(model_obj.classes_))
    elif cls_name == "SVC":
        print("kernel:", model_obj.kernel)
        print("C:", model_obj.C)
        print("gamma:", model_obj.gamma)
        print("degree:", model_obj.degree)
        print("n_features_in_:", getattr(model_obj, "n_features_in_", "?"))
        print("classes_:", list(model_obj.classes_))
    elif cls_name == "XGBClassifier":
        booster_params = model_obj.get_params()
        for key in ["max_depth", "learning_rate", "n_estimators", "subsample", "colsample_bytree"]:
            if key in booster_params:
                print(f"{key}:", booster_params[key])
        print("objective:", booster_params.get("objective"))
        print("num_class:", booster_params.get("num_class"))
    else:
        print("Unknown model type")

In [5]:
print(f"\nLoading models from: {MODELS_DIR}")
results = load_results()
models = load_all_models()

for name, model_obj in models.items():
    print_model_specs(name, model_obj)

best_model_name = results[0]["model"]
best_model_obj = models[f"{best_model_name}.pkl"] if f"{best_model_name}.pkl" in models else None
if best_model_obj is None:
    if f"{best_model_name}.json" in models:
        best_model_obj = models[f"{best_model_name}.json"]
    elif f"{best_model_name}.pt" in models:
        best_model_obj = models[f"{best_model_name}.pt"]


print(f"\nBest model: {best_model_name}")
print_model_specs(best_model_name, best_model_obj)

if isinstance(best_model_obj, tuple):
    best_model, ckpt = best_model_obj
    print("Architecture:", best_model)
    print("Hyperparameters:", ckpt["config"])
else:
    print("Model object:", best_model_obj)


Loading models from: d:\Computerpreter\Fingerspelling\models

=== MLP_1.pt ===
Type: MLP
Hidden layers: (128, 128)
Dropout: 0.15
Learning rate: 0.001
Weight decay: 0.0001
Device: cpu
Parameters: 28058

=== MLP_2.pt ===
Type: MLP
Hidden layers: (256, 256)
Dropout: 0.2
Learning rate: 0.0008
Weight decay: 0.0001
Device: cpu
Parameters: 88858

=== MLP_3.pt ===
Type: MLP
Hidden layers: (512, 256)
Dropout: 0.25
Learning rate: 0.0008
Weight decay: 5e-05
Device: cpu
Parameters: 170778

=== MLP_4.pt ===
Type: MLP
Hidden layers: (512, 512, 256)
Dropout: 0.3
Learning rate: 0.0005
Weight decay: 5e-05
Device: cpu
Parameters: 433434

=== RF_1.pkl ===
Type: RandomForestClassifier
n_estimators: 200
max_depth: None
min_samples_split: 2
n_features_in_: 63
classes_: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]

=== RF_2.pkl ===
Type: RandomForestClassifier
n_estimators: 400
max_depth: 20
min_samples_split: 2
n_features_in_: 63
classes_: [0, 1, 2, 3, 4, 5